# 01 Dataset and Embedding

Part 2: Önceki 3 harf bağlamı (context) ile X ve Y veri seti kurulumu, 27x2 embedding tablosu ve indeksleme.


In [ ]:
import torch

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(stoi)

block_size = 3
X, Y = [], []
for w in words:
    context = [0] * block_size
    for ch in w + ".":
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)
print("Vocabulary size:", vocab_size)
print("X shape:", X.shape)
print("Y shape:", Y.shape)

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)

# Indeksleme ile embedding çekme
print("Ilk ornek baglami (X[0]):", X[0])
print("Ilk ornek embeddingleri (C[X[0]]):\n", C[X[0]])
emb = C[X]
print("Tum veri embedding tensöru sekli (C[X]):", emb.shape)


# 02 Hidden Layer and Loss

Gizli katman (tanh), çıkış katmanı (logits), manuel loss hesaplama ve F.cross_entropy karşılaştırması.


In [ ]:
import torch
import torch.nn.functional as F

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

block_size = 3
X, Y = [], []
for w in words:
    context = [0] * block_size
    for ch in w + ".":
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)

# Embedding duzlestirme ve gizli katman
emb = C[X]
emb_flat = emb.view(-1, 6)
h = torch.tanh(emb_flat @ W1 + b1)
logits = h @ W2 + b2

# Manuel cross-entropy hesaplama
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
loss_manual = -probs[torch.arange(len(Y)), Y].log().mean()

# F.cross_entropy ile karsilastirma
loss_ce = F.cross_entropy(logits, Y)

print("Manuel loss:", loss_manual.item())
print("F.cross_entropy loss:", loss_ce.item())
print("Fark:", torch.abs(loss_manual - loss_ce).item())
assert torch.allclose(loss_manual, loss_ce), "Loss sonuclari ayni degil!"

# F.cross_entropy neden tercih edilir:
# 1. Numerik kararlilik: Buyuk logitlerde exp() overflow (inf/nan) olusur, cross_entropy max degeri cikarir
extreme_logits = torch.tensor([[50.0, 1000.0, 20.0]])
target = torch.tensor([1])
try:
    bad_counts = extreme_logits.exp()
    bad_probs = bad_counts / bad_counts.sum(1, keepdim=True)
    bad_loss = -bad_probs[0, target].log()
    print("Manuel asiri logit loss:", bad_loss.item())
except Exception as e:
    print("Manuel asiri logit hata:", e)

stable_loss = F.cross_entropy(extreme_logits, target)
print("F.cross_entropy kararlı loss:", stable_loss.item())
# 2. Analitik turev: Fused kernel ara tensorleri bellekte tutmaz, cok daha hizli ve az bellek harcar


# 03 Training Loop and LR Search

Tek minibatch overfit testi, learning rate taraması, train/dev/test bölme ve eğitim döngüsü.


In [ ]:
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

def build_dataset(words, block_size=3):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

# 1. Tek minibatch overfit testi
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g, requires_grad=True)
W1 = torch.randn((6, 100), generator=g, requires_grad=True)
b1 = torch.randn(100, generator=g, requires_grad=True)
W2 = torch.randn((100, 27), generator=g, requires_grad=True)
b2 = torch.randn(27, generator=g, requires_grad=True)
parameters = [C, W1, b1, W2, b2]

ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

for i in range(200):
    emb = C[Xb].view(-1, 6)
    h = torch.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)
    
    for p in parameters:
        p.grad = None
    loss.backward()
    for p in parameters:
        p.data += -0.1 * p.grad

print("Overfit edilen tek batch loss:", loss.item())

# 2. Learning rate taraması (LR Search)
C = torch.randn((27, 2), generator=g, requires_grad=True)
W1 = torch.randn((6, 100), generator=g, requires_grad=True)
b1 = torch.randn(100, generator=g, requires_grad=True)
W2 = torch.randn((100, 27), generator=g, requires_grad=True)
b2 = torch.randn(27, generator=g, requires_grad=True)
parameters = [C, W1, b1, W2, b2]

lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre
lri, lossi = [], []

for i in range(1000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    emb = C[Xtr[ix]].view(-1, 6)
    h = torch.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    
    for p in parameters:
        p.grad = None
    loss.backward()
    lr = lrs[i]
    for p in parameters:
        p.data += -lr * p.grad
    lri.append(lre[i])
    lossi.append(loss.item())

plt.figure(figsize=(8, 4))
plt.plot(lri, lossi)
plt.xlabel("log10(lr)")
plt.ylabel("loss")
plt.title("Learning Rate Finder")
plt.show()

# 3. Tum egitim seti ile egitim ve dev loss
C = torch.randn((27, 2), generator=g, requires_grad=True)
W1 = torch.randn((6, 100), generator=g, requires_grad=True)
b1 = torch.randn(100, generator=g, requires_grad=True)
W2 = torch.randn((100, 27), generator=g, requires_grad=True)
b2 = torch.randn(27, generator=g, requires_grad=True)
parameters = [C, W1, b1, W2, b2]

for i in range(30000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    emb = C[Xtr[ix]].view(-1, 6)
    h = torch.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    
    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 20000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

emb_dev = C[Xdev].view(-1, 6)
h_dev = torch.tanh(emb_dev @ W1 + b1)
logits_dev = h_dev @ W2 + b2
loss_dev = F.cross_entropy(logits_dev, Ydev)
print("Dev set loss:", loss_dev.item())


# 04 Scaled Model and Sampling

Embedding ve gizli katmanı büyütme, 2D embedding görselleştirme ve modelden isim örnekleme.


In [ ]:
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

def build_dataset(words, block_size=3):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

# 2D embedding modeli (cizim icin)
g = torch.Generator().manual_seed(2147483647)
C_2d = torch.randn((27, 2), generator=g, requires_grad=True)
W1_2d = torch.randn((6, 100), generator=g, requires_grad=True)
b1_2d = torch.randn(100, generator=g, requires_grad=True)
W2_2d = torch.randn((100, 27), generator=g, requires_grad=True)
b2_2d = torch.randn(27, generator=g, requires_grad=True)
params_2d = [C_2d, W1_2d, b1_2d, W2_2d, b2_2d]

for i in range(30000):
    ix = torch.randint(0, Xtr.shape[0], (64,))
    emb = C_2d[Xtr[ix]].view(-1, 6)
    h = torch.tanh(emb @ W1_2d + b1_2d)
    logits = h @ W2_2d + b2_2d
    loss = F.cross_entropy(logits, Ytr[ix])
    for p in params_2d:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 20000 else 0.01
    for p in params_2d:
        p.data += -lr * p.grad

emb_dev = C_2d[Xdev].view(-1, 6)
loss_2d = F.cross_entropy(torch.tanh(emb_dev @ W1_2d + b1_2d) @ W2_2d + b2_2d, Ydev)
print("2D Embedding Dev Loss:", loss_2d.item())

# 2D Embedding Görselleştirme
plt.figure(figsize=(8, 8))
plt.scatter(C_2d[:, 0].data, C_2d[:, 1].data, s=250)
for i in range(C_2d.shape[0]):
    plt.text(C_2d[i, 0].item(), C_2d[i, 1].item(), itos[i], ha="center", va="center", color="white", fontsize=11)
plt.grid(True)
plt.title("2D Karakter Embedding Uzayi")
plt.show()

# Boyutlari buyutulmus model (emb_dim=10, hidden=200)
n_emb = 10
n_hidden = 200
C_sc = torch.randn((27, n_emb), generator=g, requires_grad=True)
W1_sc = torch.randn((n_emb * 3, n_hidden), generator=g, requires_grad=True)
b1_sc = torch.randn(n_hidden, generator=g, requires_grad=True)
W2_sc = torch.randn((n_hidden, 27), generator=g, requires_grad=True)
b2_sc = torch.randn(27, generator=g, requires_grad=True)
params_sc = [C_sc, W1_sc, b1_sc, W2_sc, b2_sc]

for i in range(35000):
    ix = torch.randint(0, Xtr.shape[0], (64,))
    emb = C_sc[Xtr[ix]].view(-1, n_emb * 3)
    h = torch.tanh(emb @ W1_sc + b1_sc)
    logits = h @ W2_sc + b2_sc
    loss = F.cross_entropy(logits, Ytr[ix])
    for p in params_sc:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 25000 else 0.01
    for p in params_sc:
        p.data += -lr * p.grad

emb_dev = C_sc[Xdev].view(-1, n_emb * 3)
loss_sc = F.cross_entropy(torch.tanh(emb_dev @ W1_sc + b1_sc) @ W2_sc + b2_sc, Ydev)
print("Buyuk Model Dev Loss (10-dim, 200 hidden):", loss_sc.item())

# Modelden İsim Örnekleme (Sampling)
g_sample = torch.Generator().manual_seed(2147483647 + 10)
print("\n--- MLP Uretilen Isimler ---")
for _ in range(10):
    out = []
    context = [0] * 3
    while True:
        emb = C_sc[torch.tensor([context])].view(1, -1)
        h = torch.tanh(emb @ W1_sc + b1_sc)
        logits = h @ W2_sc + b2_sc
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g_sample).item()
        context = context[1:] + [ix]
        if ix == 0:
            break
        out.append(itos[ix])
    print("".join(out))


# 05 Initialization and Tanh Saturation

Part 3: Başlangıç loss'unun yüksekliği, tanh doyumu (saturation), ölü gradyanlar ve Kaiming Normal init.


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

block_size = 3
X, Y = [], []
for w in words:
    context = [0] * block_size
    for ch in w + ".":
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

# 1. Olceksiz baslatmada yuksek loss ve tanh doyumu
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 200), generator=g)
b1 = torch.randn(200, generator=g)
W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)

emb = C[X[:1000]].view(-1, 30)
hpreact = emb @ W1 + b1
h = torch.tanh(hpreact)
logits = h @ W2 + b2
loss_unscaled = F.cross_entropy(logits, Y[:1000])

print("Olceksiz baslangic loss:", loss_unscaled.item())
print("Beklenen baslangic loss (-log(1/27)):", -torch.log(torch.tensor(1/27)).item())
print("Doymus tanh orani (|h| > 0.99):", (h.abs() > 0.99).float().mean().item())

plt.figure(figsize=(12, 4))
plt.subplot(121)
plt.hist(hpreact.view(-1).tolist(), 50, density=True)
plt.title("Pre-aktivasyonlar (Olceksiz)")
plt.subplot(122)
plt.hist(h.view(-1).tolist(), 50, density=True)
plt.title("Tanh Aktivasyonlari (Uclarda Doymus)")
plt.show()

# 2. Kaiming Normal init ve W2 olcekleme ile cozum
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 10), generator=g)
# Tanh icin gain = 5/3, fan_in = 30
W1 = torch.randn((30, 200), generator=g) * (5/3) / (30**0.5)
b1 = torch.randn(200, generator=g) * 0.01
# W2 kucultulerek esit baslangic olasiliklari saglanir
W2 = torch.randn((200, 27), generator=g) * 0.01
b2 = torch.randn(27, generator=g) * 0

emb = C[X[:1000]].view(-1, 30)
hpreact = emb @ W1 + b1
h = torch.tanh(hpreact)
logits = h @ W2 + b2
loss_kaiming = F.cross_entropy(logits, Y[:1000])

print("Kaiming init sonrasi baslangic loss:", loss_kaiming.item())
print("Kaiming sonrasi doymus tanh orani:", (h.abs() > 0.99).float().mean().item())

plt.figure(figsize=(12, 4))
plt.subplot(121)
plt.hist(hpreact.view(-1).tolist(), 50, density=True)
plt.title("Pre-aktivasyonlar (Kaiming Init)")
plt.subplot(122)
plt.hist(h.view(-1).tolist(), 50, density=True)
plt.title("Tanh Aktivasyonlari (Duzenli Dagilim)")
plt.show()


# 06 Batch Normalization

Part 3: BatchNorm katmanı implementasyonu, eğitimde batch istatistiği, çıkarımda running mean/std, BatchNorm'lu vs BatchNorm'suz karşılaştırma.


In [ ]:
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

def build_dataset(words, block_size=3):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])

n_emb = 10
n_hidden = 200

# 1. BatchNorm'lu Model
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, n_emb), generator=g, requires_grad=True)
W1 = torch.randn((n_emb * 3, n_hidden), generator=g) * (5/3) / ((n_emb * 3)**0.5)
W1.requires_grad = True
# b1 BatchNorm ortalamayi cikardigi icin gereksizdir
W2 = torch.randn((n_hidden, 27), generator=g) * 0.01
W2.requires_grad = True
b2 = torch.randn(27, generator=g) * 0
b2.requires_grad = True

bngain = torch.ones((1, n_hidden), requires_grad=True)
bnbias = torch.zeros((1, n_hidden), requires_grad=True)
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters_bn = [C, W1, W2, b2, bngain, bnbias]

# 2. BatchNorm'suz Model (Baseline)
g_nb = torch.Generator().manual_seed(2147483647)
C_nb = torch.randn((27, n_emb), generator=g_nb, requires_grad=True)
W1_nb = torch.randn((n_emb * 3, n_hidden), generator=g_nb) * (5/3) / ((n_emb * 3)**0.5)
W1_nb.requires_grad = True
b1_nb = torch.randn(n_hidden, generator=g_nb) * 0.01
b1_nb.requires_grad = True
W2_nb = torch.randn((n_hidden, 27), generator=g_nb) * 0.01
W2_nb.requires_grad = True
b2_nb = torch.randn(27, generator=g_nb) * 0
b2_nb.requires_grad = True

parameters_nb = [C_nb, W1_nb, b1_nb, W2_nb, b2_nb]

for i in range(20000):
    ix = torch.randint(0, Xtr.shape[0], (64,))
    
    # BatchNorm ileri gecis
    emb = C[Xtr[ix]].view(-1, n_emb * 3)
    hpreact = emb @ W1
    bnmean = hpreact.mean(0, keepdim=True)
    bnstd = hpreact.std(0, keepdim=True)
    hpreact_norm = (hpreact - bnmean) / (bnstd + 1e-5) * bngain + bnbias
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmean
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstd
    h = torch.tanh(hpreact_norm)
    logits = h @ W2 + b2
    loss_bn = F.cross_entropy(logits, Ytr[ix])
    
    for p in parameters_bn:
        p.grad = None
    loss_bn.backward()
    lr = 0.1 if i < 15000 else 0.01
    for p in parameters_bn:
        p.data += -lr * p.grad
        
    # BatchNorm'suz ileri gecis
    emb_nb = C_nb[Xtr[ix]].view(-1, n_emb * 3)
    h_nb = torch.tanh(emb_nb @ W1_nb + b1_nb)
    logits_nb = h_nb @ W2_nb + b2_nb
    loss_nb = F.cross_entropy(logits_nb, Ytr[ix])
    
    for p in parameters_nb:
        p.grad = None
    loss_nb.backward()
    for p in parameters_nb:
        p.data += -lr * p.grad

# Dev seti degerlendirmesi (cikarimda running istatistikler kullanilir)
emb_dev = C[Xdev].view(-1, n_emb * 3)
hpreact_dev = emb_dev @ W1
hpreact_dev_norm = (hpreact_dev - bnmean_running) / (bnstd_running + 1e-5) * bngain + bnbias
h_dev = torch.tanh(hpreact_dev_norm)
logits_dev = h_dev @ W2 + b2
dev_loss_bn = F.cross_entropy(logits_dev, Ydev)

emb_dev_nb = C_nb[Xdev].view(-1, n_emb * 3)
h_dev_nb = torch.tanh(emb_dev_nb @ W1_nb + b1_nb)
logits_dev_nb = h_dev_nb @ W2_nb + b2_nb
dev_loss_nb = F.cross_entropy(logits_dev_nb, Ydev)

print("BatchNorm ile Dev Loss:", dev_loss_bn.item())
print("BatchNorm olmadan Dev Loss:", dev_loss_nb.item())


# 07 Turkish Names MLP

Türkçe isimler veri kümesi ile MLP modelinin eğitimi, dev loss değerlendirmesi ve Bigram modeliyle karşılaştırmalı örnekleme.


In [ ]:
import csv
import io
import unicodedata
from urllib.request import urlopen
import random
import torch
import torch.nn.functional as F

url = "https://raw.githubusercontent.com/niyazikemer/turkce_isimler/main/turkce_isim.csv"
text = urlopen(url).read().decode("utf-8-sig")
rows = csv.DictReader(io.StringIO(text))
words = sorted(set(unicodedata.normalize("NFC", row["name"].strip().lower()) for row in rows))
words = [w for w in words if w and w.isalpha()]

chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(stoi)
print("Turkce alfabe boyutu (vocab_size):", vocab_size)
print("Toplam Turkce isim:", len(words))

block_size = 3
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

# Turkce MLP Modeli (Kaiming init + BatchNorm)
n_emb = 10
n_hidden = 200
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_emb), generator=g, requires_grad=True)
W1 = torch.randn((n_emb * block_size, n_hidden), generator=g) * (5/3) / ((n_emb * block_size)**0.5)
W1.requires_grad = True
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
W2.requires_grad = True
b2 = torch.randn(vocab_size, generator=g) * 0
b2.requires_grad = True

bngain = torch.ones((1, n_hidden), requires_grad=True)
bnbias = torch.zeros((1, n_hidden), requires_grad=True)
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, W2, b2, bngain, bnbias]

for i in range(25000):
    ix = torch.randint(0, Xtr.shape[0], (64,))
    emb = C[Xtr[ix]].view(-1, n_emb * block_size)
    hpreact = emb @ W1
    bnmean = hpreact.mean(0, keepdim=True)
    bnstd = hpreact.std(0, keepdim=True)
    hpreact_norm = (hpreact - bnmean) / (bnstd + 1e-5) * bngain + bnbias
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmean
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstd
    h = torch.tanh(hpreact_norm)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    
    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 18000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

# Dev Loss
emb_dev = C[Xdev].view(-1, n_emb * block_size)
hpreact_dev = emb_dev @ W1
hpreact_dev_norm = (hpreact_dev - bnmean_running) / (bnstd_running + 1e-5) * bngain + bnbias
h_dev = torch.tanh(hpreact_dev_norm)
logits_dev = h_dev @ W2 + b2
dev_loss = F.cross_entropy(logits_dev, Ydev)
print("Turkce MLP Dev Loss:", dev_loss.item())

# Karsilastirma icin Bigram Modeli
N_bigram = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)
for w in words[:n1]:
    chs = ["."] + list(w) + ["."]
    for ch1, ch2 in zip(chs, chs[1:]):
        N_bigram[stoi[ch1], stoi[ch2]] += 1
P_bigram = (N_bigram + 1).float()
P_bigram /= P_bigram.sum(1, keepdim=True)

# Yan Yana Ornekleme (Bigram vs MLP)
g_sample = torch.Generator().manual_seed(42)
print("\n" + "="*50)
print(f"{'No':<4} {'Bigram Uretimi':<20} {'MLP Uretimi':<20}")
print("="*50)

for idx in range(10):
    # Bigram
    out_bi = []
    ix_bi = 0
    while True:
        ix_bi = torch.multinomial(P_bigram[ix_bi], 1, replacement=True, generator=g_sample).item()
        if ix_bi == 0:
            break
        out_bi.append(itos[ix_bi])
        
    # MLP
    out_mlp = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])].view(1, -1)
        hpreact = emb @ W1
        hpreact_norm = (hpreact - bnmean_running) / (bnstd_running + 1e-5) * bngain + bnbias
        h = torch.tanh(hpreact_norm)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix_mlp = torch.multinomial(probs, num_samples=1, generator=g_sample).item()
        context = context[1:] + [ix_mlp]
        if ix_mlp == 0:
            break
        out_mlp.append(itos[ix_mlp])
        
    print(f"{idx+1:<4} {''.join(out_bi):<20} {''.join(out_mlp):<20}")


# 08 Exercises

Part 3 E02 (BatchNorm'u önceki Linear katmana katlama / folding) ve Part 2 E01 (Hiperparametre optimizasyonu ile validation loss < 2.2).


In [ ]:
import random
import torch
import torch.nn.functional as F

# 1. Part 3 E02: BatchNorm'u Linear Katmanın W ve b'sine katlama (Folding)
words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

block_size = 3
n_emb = 10
n_hidden = 100

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, n_emb), generator=g)
W1 = torch.randn((n_emb * block_size, n_hidden), generator=g) * 0.2
b1 = torch.randn(n_hidden, generator=g) * 0.1
bngain = torch.randn((1, n_hidden), generator=g) * 0.5 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g) * 0.2
bnmean_running = torch.randn((1, n_hidden), generator=g) * 0.1
bnstd_running = torch.rand((1, n_hidden), generator=g) + 0.5
eps = 1e-5

# Test girdisi
X_sample = torch.randint(0, 27, (10, block_size))
emb = C[X_sample].view(-1, n_emb * block_size)

# Standart BatchNorm ileri gecis:
# hpreact_norm = (emb @ W1 + b1 - bnmean_running) / (bnstd_running + eps) * bngain + bnbias
linear_out = emb @ W1 + b1
bn_out = (linear_out - bnmean_running) / (bnstd_running + eps) * bngain + bnbias

# Katlanmis Linear agirlik ve bias:
# (X @ W1 + b1 - mu) * (gamma / sigma) + beta = X @ [W1 * (gamma / sigma)] + [(b1 - mu) * (gamma / sigma) + beta]
scale = bngain / (bnstd_running + eps)
W_folded = W1 * scale
b_folded = (b1 - bnmean_running) * scale + bnbias

folded_out = emb @ W_folded + b_folded

max_diff = (bn_out - folded_out).abs().max().item()
print("BatchNorm vs Katlanmis Linear maksimum cikis farki:", max_diff)
assert max_diff < 1e-5, "Folding dogrulamasi basarisiz!"
print("Dogrulama basarili: BatchNorm parametreleri Linear katmana kusursuzca katlandi.")

# 2. Part 2 E01: Hiperparametreleri ayarlayarak Karpathy 2.2 val loss hedefini gecme
def build_dataset(words, block_size=3):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1], block_size=3)
Xdev, Ydev = build_dataset(words[n1:n2], block_size=3)

n_emb = 10
n_hidden = 200
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, n_emb), generator=g, requires_grad=True)
W1 = torch.randn((n_emb * 3, n_hidden), generator=g) * (5/3) / ((n_emb * 3)**0.5)
W1.requires_grad = True
W2 = torch.randn((n_hidden, 27), generator=g) * 0.01
W2.requires_grad = True
b2 = torch.randn(27, generator=g) * 0
b2.requires_grad = True
bngain = torch.ones((1, n_hidden), requires_grad=True)
bnbias = torch.zeros((1, n_hidden), requires_grad=True)
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

params = [C, W1, W2, b2, bngain, bnbias]

for i in range(35000):
    ix = torch.randint(0, Xtr.shape[0], (64,))
    emb = C[Xtr[ix]].view(-1, n_emb * 3)
    hpreact = emb @ W1
    bnmean = hpreact.mean(0, keepdim=True)
    bnstd = hpreact.std(0, keepdim=True)
    hpreact_norm = (hpreact - bnmean) / (bnstd + 1e-5) * bngain + bnbias
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmean
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstd
    h = torch.tanh(hpreact_norm)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    
    for p in params:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 25000 else 0.01
    for p in params:
        p.data += -lr * p.grad

emb_dev = C[Xdev].view(-1, n_emb * 3)
hpreact_dev = emb_dev @ W1
hpreact_dev_norm = (hpreact_dev - bnmean_running) / (bnstd_running + 1e-5) * bngain + bnbias
h_dev = torch.tanh(hpreact_dev_norm)
logits_dev = h_dev @ W2 + b2
val_loss = F.cross_entropy(logits_dev, Ydev).item()
print(f"Validation loss: {val_loss:.4f} (Karpathy 2.2 hedefini gecti mi: {val_loss < 2.2})")
